# Python 05 — Slicing for Production Data Handling

**Roadmap position:** Python → Core → Slicing.

Slicing is useful in backend and data-processing work: serving one page of in-memory records, showing a bounded recent-log view, sampling a sequence, and masking sensitive identifiers. Use it deliberately: a slice is usually a new sequence, so very large slices have a cost.

## Outcome

You will calculate slice boundaries correctly, use omitted bounds and steps, validate pagination inputs, and debug off-by-one errors.

## 1. Slice mechanics

The form is `sequence[start:stop:step]`. `start` is included; `stop` is excluded. That matches `range(start, stop)`. Omitted bounds mean the beginning or end of the sequence. Negative indexes count from the end.

For a list or string, slicing does not mutate the original sequence. It returns a new list or string.

In [ ]:
request_ids = ['req-100', 'req-101', 'req-102', 'req-103', 'req-104']

print(request_ids[1:4])   # Indexes 1, 2, 3
print(request_ids[:2])    # First two
print(request_ids[-2:])   # Last two
print(request_ids[::2])   # Every second item
print(request_ids)        # Original is unchanged

## 2. In-memory pagination

For a one-indexed page number and page size:

```text
start index = (page number - 1) × page size
stop index  = start index + page size
```

This is suitable for a small in-memory collection. A production database with many rows commonly uses database-level pagination, often cursor-based pagination, to avoid repeatedly scanning and discarding earlier rows.

In [ ]:
def paginate_records(
    records: list[str],
    page_number: int,
    page_size: int,
) -> list[str]:
    """Return one one-indexed page from a small in-memory record list."""
    if page_number <= 0:
        raise ValueError('page_number must be greater than zero')
    if page_size <= 0:
        raise ValueError('page_size must be greater than zero')

    start_index = (page_number - 1) * page_size
    stop_index = start_index + page_size
    return records[start_index:stop_index]


application_ids = ['app-1', 'app-2', 'app-3', 'app-4', 'app-5']
print(paginate_records(application_ids, page_number=2, page_size=2))

## Your turn 1 — paginate API results

Implement `paginate_job_ids(job_ids: list[str], page_number: int, page_size: int) -> list[str]`. Follow the same contract as the example above.

Requirements:

- page numbering starts at 1;
- raise `ValueError` for a page number or page size less than 1;
- return an empty list for a valid page beyond the available items;
- do not modify `job_ids`.

In [ ]:
# YOUR TURN
def paginate_job_ids(
    job_ids: list[str],
    page_number: int,
    page_size: int,
) -> list[str]:
    raise NotImplementedError


In [ ]:
# Checks — run after implementing paginate_job_ids.
job_ids = ['job-1', 'job-2', 'job-3', 'job-4', 'job-5']
assert paginate_job_ids(job_ids, 1, 2) == ['job-1', 'job-2']
assert paginate_job_ids(job_ids, 2, 2) == ['job-3', 'job-4']
assert paginate_job_ids(job_ids, 3, 2) == ['job-5']
assert paginate_job_ids(job_ids, 4, 2) == []
assert job_ids == ['job-1', 'job-2', 'job-3', 'job-4', 'job-5']

for invalid_arguments in ((0, 2), (1, 0), (-1, 1)):
    try:
        paginate_job_ids(job_ids, *invalid_arguments)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{invalid_arguments} should raise ValueError')

print('Pagination checks passed.')

## 3. Bounded log views

Operational dashboards should not send every event to a user. A bounded view such as the last 100 log lines prevents excessive response size and keeps the newest information visible.

A negative slice start is useful here: `events[-limit:]` asks for up to the final `limit` items. If there are fewer items, Python safely returns all of them.

In [ ]:
recent_events = [
    'INFO app started',
    'INFO database connected',
    'WARNING slow query',
    'ERROR upstream timeout',
]

print(recent_events[-2:])
print(recent_events)  # The original list is unchanged.

## Your turn 2 — expose a safe recent-event view

Implement `recent_events(events: list[str], limit: int) -> list[str]`. Return at most the last `limit` events. Raise `ValueError` when `limit` is less than 1, and do not mutate `events`.

Use one slice for the returned value. Do not write a loop.

In [ ]:
# YOUR TURN
def recent_events(events: list[str], limit: int) -> list[str]:
    raise NotImplementedError


In [ ]:
# Checks — run after implementing recent_events.
events = ['event-1', 'event-2', 'event-3']
assert recent_events(events, 2) == ['event-2', 'event-3']
assert recent_events(events, 10) == ['event-1', 'event-2', 'event-3']
assert events == ['event-1', 'event-2', 'event-3']

try:
    recent_events(events, 0)
except ValueError:
    pass
else:
    raise AssertionError('limit=0 should raise ValueError')

print('Recent-event checks passed.')

## 4. String slicing and safe display

Avoid logging secrets. When an interface needs to distinguish a token, show only a small suffix. Slicing is appropriate for display masking, but it is not encryption and must not be treated as a security control.

Never write a full access token, password, or API key to application logs.

In [ ]:
def mask_identifier(identifier: str, visible_suffix_length: int = 4) -> str:
    """Return a display-only masked identifier."""
    if visible_suffix_length < 0:
        raise ValueError('visible_suffix_length must not be negative')
    if visible_suffix_length == 0:
        return '*' * len(identifier)
    if len(identifier) <= visible_suffix_length:
        return '*' * len(identifier)
    return '*' * (len(identifier) - visible_suffix_length) + identifier[-visible_suffix_length:]

print(mask_identifier('candidate-12345'))

## Debugging drill — off-by-one pagination

This function is meant to return the first page of two records when called with `page_number=1` and `page_size=2`. It incorrectly returns the second page.

1. Calculate its current start index for page 1.
2. State the correct formula for a one-indexed page number.
3. Make the smallest code change necessary.

**Interview question:** Why are off-by-one errors common in APIs that expose page numbers?

In [ ]:
# DEBUG ME
def broken_paginate(records: list[str], page_number: int, page_size: int) -> list[str]:
    start_index = page_number * page_size
    stop_index = start_index + page_size
    return records[start_index:stop_index]


records = ['record-1', 'record-2', 'record-3', 'record-4']
print(broken_paginate(records, page_number=1, page_size=2))  # Expected: record-1, record-2

## Exit interview check

Answer without running code:

1. Which indexes does `items[2:5]` include?
2. Why can `items[-10:]` safely be used when `items` has fewer than 10 values?
3. What is the one-indexed pagination formula, and why is it useful?
4. Does slicing a list change the original list? What resource cost should you consider for a very large slice?
5. Why might cursor-based database pagination be preferable to slicing an in-memory list for a very large API result set?

When finished, send me one completed exercise or debugging attempt at a time. I’ll review it without modifying your notebook. Then we will proceed to **comprehensions**.